# Generators

Split from `01_iterators_and_generators.ipynb` to keep this topic self-contained.

**Navigation:** [Previous: Iterators](./01_iterators.ipynb) · [Topic overview](./01_iterators_and_generators.ipynb)


# Iterators and Generators for Bioinformatics

## Learning Objectives
- Understand the iterator protocol (`__iter__`, `__next__`)
- Use `iter()` and `next()` to manually drive iteration
- Create generators with `yield` and generator expressions
- Leverage the `itertools` module for combinatorics and data processing
- Apply lazy evaluation for memory-efficient processing of large biological datasets

**Why this matters in bioinformatics:** Genome sequences, FASTQ files, and protein databases can be gigabytes in size. Iterators and generators let you process these datasets line by line without loading everything into memory.

---

[← Previous: Module 10: Comprehensions](../../Tier_1_Python_for_Bioinformatics/10_Comprehensions/01_comprehensions.ipynb) | [Next: Regular Expressions for Bioinformatics →](../../Tier_1_Python_for_Bioinformatics/12_Regular_Expressions/01_regular_expressions.ipynb)

---

---
## 1. The Iteration Protocol

In Python, any object that implements two methods is an **iterator**:
- `__iter__()` -- returns the iterator object itself
- `__next__()` -- returns the next value, or raises `StopIteration` when exhausted

An **iterable** is any object that can return an iterator via `__iter__()` (strings, lists, files, etc.).

In [ ]:
# Any iterable can be turned into an iterator with iter()
nucleotides = ["A", "T", "G", "C"]
nuc_iter = iter(nucleotides)

print(type(nucleotides))   # list -- iterable
print(type(nuc_iter))      # list_iterator -- iterator

In [ ]:
# Manually advance the iterator with next()
print(next(nuc_iter))  # A
print(next(nuc_iter))  # T
print(next(nuc_iter))  # G
print(next(nuc_iter))  # C

In [ ]:
# When an iterator is exhausted, StopIteration is raised
try:
    print(next(nuc_iter))
except StopIteration:
    print("Iterator exhausted! No more nucleotides.")

In [ ]:
# Strings are iterable too -- iterate over a DNA sequence
dna = "ATGCGA"
seq_iter = iter(dna)

print(next(seq_iter))  # A
print(next(seq_iter))  # T
print(next(seq_iter))  # G

### How `for` Loops Really Work

A `for` loop is syntactic sugar around `iter()` and `next()`:

In [ ]:
# This for loop:
for nuc in "ATGC":
    print(nuc, end=" ")
print()

# Is equivalent to this:
iterator = iter("ATGC")
while True:
    try:
        nuc = next(iterator)
        print(nuc, end=" ")
    except StopIteration:
        break
print()

### Building a Custom Iterator Class

Let's build a **CodonIterator** that walks through a DNA sequence in triplets:

In [ ]:
class CodonIterator:
    """Iterate over a DNA sequence in codons (triplets)."""
    
    def __init__(self, sequence):
        self.sequence = sequence
        self.index = 0
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.index + 3 > len(self.sequence):
            raise StopIteration
        codon = self.sequence[self.index:self.index + 3]
        self.index += 3
        return codon


dna = "ATGAAACCCGGGTTTAAA"
print(f"Sequence: {dna}")
print("Codons:")
for codon in CodonIterator(dna):
    print(f"  {codon}")

In [ ]:
# Iterators are single-pass -- once exhausted, they are done
ci = CodonIterator("ATGAAA")

print("First pass:", list(ci))
print("Second pass:", list(ci))  # empty -- already exhausted

---
## 2. Generators with `yield`

Writing a full iterator class is verbose. **Generators** are functions that use `yield` instead of `return`. Each call to `next()` resumes execution right after the last `yield`.

Generators automatically implement the iterator protocol.

In [ ]:
def codon_generator(sequence):
    """Yield codons (triplets) from a DNA sequence."""
    for i in range(0, len(sequence) - 2, 3):
        yield sequence[i:i+3]


gen = codon_generator("ATGAAACCCGGGTTT")
print(f"Type: {type(gen)}")
print(f"First codon: {next(gen)}")
print(f"Second codon: {next(gen)}")
print(f"Remaining: {list(gen)}")

In [ ]:
# Generator for k-mers (subsequences of length k)
def kmer_generator(sequence, k):
    """Yield all k-mers from a sequence."""
    for i in range(len(sequence) - k + 1):
        yield sequence[i:i+k]


dna = "ATGCGATCG"
print(f"All 3-mers from {dna}:")
print(list(kmer_generator(dna, 3)))

In [ ]:
# Generators can be infinite -- Fibonacci example for modeling populations
def fibonacci():
    """Infinite Fibonacci generator (e.g., rabbit population growth model)."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


fib = fibonacci()
print("First 12 Fibonacci numbers:")
for _ in range(12):
    print(next(fib), end=" ")
print()

In [ ]:
# Translation generator -- yield amino acids one at a time
CODON_TABLE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}


def translate_generator(sequence):
    """Lazily translate a DNA sequence into amino acids, stopping at stop codon."""
    for codon in codon_generator(sequence):
        aa = CODON_TABLE.get(codon, 'X')
        if aa == '*':
            return  # StopIteration
        yield aa


coding_seq = "ATGAAAGCCTTTGGGTGA"
protein = ''.join(translate_generator(coding_seq))
print(f"DNA:     {coding_seq}")
print(f"Protein: {protein}")

---
## 3. Generator Expressions

Generator expressions look like list comprehensions but use `()` instead of `[]`. They produce values lazily.

In [ ]:
dna = "ATGCGATCGATCGATCG"

# List comprehension -- all values computed and stored in memory
gc_list = [1 for nuc in dna if nuc in 'GC']

# Generator expression -- values computed on demand
gc_gen = (1 for nuc in dna if nuc in 'GC')

print(f"List: {gc_list}")
print(f"Generator: {gc_gen}")
print(f"GC count via generator: {sum(1 for nuc in dna if nuc in 'GC')}")

In [ ]:
# GC content with a generator expression
def gc_content(seq):
    """Calculate GC content using a generator expression."""
    return sum(1 for nuc in seq if nuc in 'GC') / len(seq)


sequences = {
    'AT-rich promoter': 'AAATTTATTTAAAGCTA',
    'CpG island':       'GCGCGGCGCGCGCCGCG',
    'Random':           'ATGCGATCGATCGATCG'
}

for name, seq in sequences.items():
    print(f"{name:20s}: GC = {gc_content(seq):.1%}")

---
## 4. Memory Efficiency

Generators are critical when working with data that does not fit in memory.

In [ ]:
import sys

# Compare memory usage: list vs generator
dna = "A" * 1_000_000  # 1 million bases

# All 3-mers as a list
kmers_list = [dna[i:i+3] for i in range(len(dna) - 2)]

# All 3-mers as a generator
kmers_gen = kmer_generator(dna, 3)

print(f"List of {len(kmers_list):,} k-mers: {sys.getsizeof(kmers_list):,} bytes")
print(f"Generator object:              {sys.getsizeof(kmers_gen)} bytes")
print(f"\nThe generator uses ~{sys.getsizeof(kmers_list) / sys.getsizeof(kmers_gen):,.0f}x less memory!")

In [ ]:
# Chaining generators for pipeline processing (no intermediate lists)
dna = "ATGAAAGCCTTTGGGTGATCGATCG" * 100

# Pipeline: codons -> amino acids -> only charged residues
codons = codon_generator(dna)
amino_acids = (CODON_TABLE.get(c, 'X') for c in codons)
charged = (aa for aa in amino_acids if aa in 'DEKRH')

# Nothing has been computed yet! Only when we consume:
charged_count = sum(1 for _ in charged)
print(f"Charged residues in {len(dna)} bp sequence: {charged_count}")

---
## 5. Bioinformatics Application: Streaming FASTQ Reader

FASTQ files can be tens of gigabytes. A generator reads one record at a time.

In [ ]:
# First, create a sample FASTQ file
fastq_content = """@SEQ_ID_001 length=30
ATGCGATCGATCGATCGATCGATCGATCGA
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@SEQ_ID_002 length=30
GCTAGCTAGCTAGCTAGCTAGCTAGCTANN
+
IIIIIIIIIIIIIIIIIIIIIIIIIII!!!
@SEQ_ID_003 length=30
TTAACCGGTTAACCGGTTAACCGGTTAACC
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@SEQ_ID_004 length=30
GGCCAATTGGCCAATTGGCCAATTGGCCAA
+
IIIIIIIIIIIIIIIIIII!!!!!!!!!!!
"""

with open('sample.fastq', 'w') as f:
    f.write(fastq_content)

print("Created sample.fastq")

In [ ]:
def read_fastq(filename):
    """Streaming FASTQ reader -- yields one record at a time.
    
    Each record is a dict with keys: 'id', 'sequence', 'quality'.
    Memory usage is constant regardless of file size.
    """
    with open(filename) as f:
        while True:
            header = f.readline().strip()
            if not header:
                break
            sequence = f.readline().strip()
            f.readline()  # '+' line
            quality = f.readline().strip()
            
            yield {
                'id': header[1:].split()[0],
                'sequence': sequence,
                'quality': quality
            }


# Process without loading entire file
print("Streaming FASTQ records:")
for record in read_fastq('sample.fastq'):
    avg_qual = sum(ord(c) - 33 for c in record['quality']) / len(record['quality'])
    gc = gc_content(record['sequence'])
    print(f"  {record['id']}: GC={gc:.1%}, mean_qual={avg_qual:.1f}")

In [ ]:
# Generator pipeline: filter low-quality reads
def quality_filter(records, min_avg_quality=30):
    """Yield only records with average quality >= threshold."""
    for record in records:
        avg_qual = sum(ord(c) - 33 for c in record['quality']) / len(record['quality'])
        if avg_qual >= min_avg_quality:
            yield record


def trim_n_bases(records):
    """Trim trailing N bases from sequences."""
    for record in records:
        seq = record['sequence'].rstrip('N')
        qual = record['quality'][:len(seq)]
        yield {**record, 'sequence': seq, 'quality': qual}


# Chain generators into a processing pipeline
raw_reads = read_fastq('sample.fastq')
trimmed = trim_n_bases(raw_reads)
filtered = quality_filter(trimmed, min_avg_quality=25)

print("Filtered and trimmed reads:")
for record in filtered:
    print(f"  {record['id']}: {record['sequence'][:40]}... (len={len(record['sequence'])})")

---
## 6. Bioinformatics Application: Streaming FASTA Reader

In [ ]:
# Create a sample FASTA file
fasta_content = """>gene1 BRCA1 tumor suppressor
ATGAAAGCCTTTGGGATCGATCGATCGATCG
ATCGATCGATCGATCGATCGATCGATCGTAA
>gene2 TP53 guardian of the genome
ATGGCCCCCGGGATCGATCGATCGATCGATC
GATCGATCGATCGATCGATCGATCGATCTGA
>gene3 EGFR growth factor receptor
ATGCCCAAATTTGGGATCGATCGATCGATCG
ATCGATCGATCGATCGATCGATCGATCGTAG
"""

with open('sample.fasta', 'w') as f:
    f.write(fasta_content)

print("Created sample.fasta")

In [ ]:
def read_fasta(filename):
    """Memory-efficient FASTA parser. Yields (header, sequence) tuples."""
    header = None
    seq_parts = []
    
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if header is not None:
                    yield (header, ''.join(seq_parts))
                header = line[1:]
                seq_parts = []
            else:
                seq_parts.append(line)
    
    if header is not None:
        yield (header, ''.join(seq_parts))


print("Parsed FASTA records:")
for header, seq in read_fasta('sample.fasta'):
    gene_id = header.split()[0]
    print(f"  {gene_id}: {len(seq)} bp, GC={gc_content(seq):.1%}")

---
## 7. Bioinformatics Application: All Possible k-mers

Generate all possible DNA k-mers of a given length (4^k total).

In [ ]:
import itertools


def all_possible_kmers(k):
    """Generator for all possible DNA k-mers of length k."""
    for combo in itertools.product('ATGC', repeat=k):
        yield ''.join(combo)


# All possible dimers (4^2 = 16)
print("All 16 possible 2-mers:")
print(list(all_possible_kmers(2)))

# How many 5-mers exist? (4^5 = 1024)
count = sum(1 for _ in all_possible_kmers(5))
print(f"\nTotal possible 5-mers: {count}")

In [ ]:
# All 64 codons grouped by amino acid
from collections import defaultdict

aa_codons = defaultdict(list)
for codon in all_possible_kmers(3):
    aa = CODON_TABLE.get(codon, '?')
    aa_codons[aa].append(codon)

print("Genetic code degeneracy:")
for aa in sorted(aa_codons, key=lambda a: len(aa_codons[a]), reverse=True):
    codons = aa_codons[aa]
    label = 'Stop' if aa == '*' else aa
    print(f"  {label}: {len(codons)} codons -- {', '.join(codons)}")

---
## 8. Bioinformatics Application: Lazy Codon Iterator over a Genome

When processing an entire genome, we want to iterate over codons in all three reading frames without storing them.

In [ ]:
def genome_codons(sequence, frame=0):
    """Lazily iterate over codons in a specific reading frame.
    
    Args:
        sequence: DNA sequence (can be very long)
        frame: reading frame offset (0, 1, or 2)
    """
    for i in range(frame, len(sequence) - 2, 3):
        yield sequence[i:i+3]


def six_frame_translation(sequence):
    """Generator that yields (strand, frame, position, codon, aa) for all 6 frames."""
    complement = str.maketrans('ATGC', 'TACG')
    rev_comp = sequence.translate(complement)[::-1]
    
    for strand, seq in [('+', sequence), ('-', rev_comp)]:
        for frame in range(3):
            for i, codon in enumerate(genome_codons(seq, frame)):
                aa = CODON_TABLE.get(codon, 'X')
                pos = frame + i * 3
                yield (strand, frame + 1, pos, codon, aa)


# Demonstrate with a short sequence
test_seq = "ATGAAACCCGGGTTT"
print(f"Sequence: {test_seq}\n")

# Show first few codons from each frame
from itertools import groupby

results = list(six_frame_translation(test_seq))
for (strand, frame), group in groupby(results, key=lambda x: (x[0], x[1])):
    items = list(group)
    codons = ' '.join(f"{c[3]}({c[4]})" for c in items[:4])
    print(f"  Frame {strand}{frame}: {codons}")

---
## 9. The `itertools` Module

The `itertools` module provides high-performance building blocks for iterator-based programming.

### 9.1 Infinite Iterators: `count`, `cycle`, `repeat`

In [ ]:
# count -- infinite counter
# Useful for numbering positions in a sequence
from itertools import count

dna = "ATGCGA"
print("Position numbering (1-based):")
for pos, nuc in zip(count(1), dna):
    print(f"  Position {pos}: {nuc}")

In [ ]:
# cycle -- repeat a sequence forever
# Assign reading frames to positions in a sequence
from itertools import cycle

dna = "ATGAAACCCGGGTTT"
frames = cycle([1, 2, 3])

print("Reading frame assignment:")
for nuc, frame in zip(dna, frames):
    print(f"  {nuc} -> frame {frame}")

In [ ]:
# repeat -- repeat a value n times
# Generate a polyA tail
from itertools import repeat

poly_a_tail = ''.join(repeat('A', 200))
print(f"Poly-A tail: {poly_a_tail[:50]}... (length={len(poly_a_tail)})")

### 9.2 `chain` -- Combine Multiple Iterables

In [ ]:
from itertools import chain

# Concatenate gene fragments with a linker
exon1 = "ATGAAA"
linker = "NNN"
exon2 = "CCCGGG"
exon3 = "TTTTGA"

construct = ''.join(chain(exon1, linker, exon2, linker, exon3))
print(f"Gene construct: {construct}")

# chain.from_iterable -- flatten a list of sequences
exons = ["ATGAAA", "CCCGGG", "TTTTGA"]
all_bases = list(chain.from_iterable(exons))
print(f"All bases from exons: {''.join(all_bases)}")

### 9.3 `combinations` and `permutations`

In [ ]:
from itertools import combinations, permutations

# All pairwise comparisons of 4 species
species = ['Human', 'Mouse', 'Zebrafish', 'Drosophila']

print("Pairwise species comparisons:")
for sp1, sp2 in combinations(species, 2):
    print(f"  {sp1} vs {sp2}")

print(f"\nTotal pairs: {sum(1 for _ in combinations(species, 2))}")

In [ ]:
# Permutations of nucleotides -- useful for degenerate codon analysis
print("All orderings of 'ATG':")
for perm in permutations('ATG'):
    print(f"  {''.join(perm)}", end="")
print()

# How many ways can we arrange 4 samples in a gel?
samples = ['S1', 'S2', 'S3', 'S4']
n_arrangements = sum(1 for _ in permutations(samples))
print(f"\nArrangements of {len(samples)} samples: {n_arrangements}")

### 9.4 `product` -- Cartesian Product

In [ ]:
from itertools import product

# Generate all possible codons (4^3 = 64)
all_codons = [''.join(c) for c in product('ATGC', repeat=3)]
print(f"Total codons: {len(all_codons)}")
print(f"First 8: {all_codons[:8]}")
print(f"Last 8:  {all_codons[-8:]}")

In [ ]:
# Design all possible PCR primer pairs from two sets
forward_primers = ['ATGCGA', 'ATGCGC', 'ATGCGT']
reverse_primers = ['GCTAGC', 'GCTAGG']

print("All primer pair combinations:")
for fwd, rev in product(forward_primers, reverse_primers):
    print(f"  Forward: {fwd}  Reverse: {rev}")

### 9.5 `groupby` -- Group Consecutive Elements

In [ ]:
from itertools import groupby

# Find homopolymer runs in a sequence
dna = "AAATTTGGGCCAATTGCCCCCCC"

print(f"Sequence: {dna}")
print("\nHomopolymer runs:")
for nucleotide, group in groupby(dna):
    run = list(group)
    length = len(run)
    label = " <-- long run!" if length >= 5 else ""
    print(f"  {nucleotide} x {length}{label}")

In [ ]:
# Group amino acids by property
AA_PROPERTY = {
    'A': 'nonpolar', 'V': 'nonpolar', 'L': 'nonpolar', 'I': 'nonpolar',
    'P': 'nonpolar', 'F': 'nonpolar', 'W': 'nonpolar', 'M': 'nonpolar',
    'G': 'nonpolar',
    'S': 'polar', 'T': 'polar', 'C': 'polar', 'Y': 'polar',
    'N': 'polar', 'Q': 'polar',
    'D': 'negative', 'E': 'negative',
    'K': 'positive', 'R': 'positive', 'H': 'positive'
}

protein = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTK"
properties = [(aa, AA_PROPERTY.get(aa, 'unknown')) for aa in protein]

print(f"Protein: {protein}")
print("\nConsecutive stretches by property:")
for prop, group in groupby(properties, key=lambda x: x[1]):
    residues = ''.join(aa for aa, _ in group)
    if len(residues) >= 3:
        print(f"  {prop:10s}: {residues}")

### 9.6 `islice`, `takewhile`, `dropwhile`

In [ ]:
from itertools import islice, takewhile, dropwhile

# islice -- slice an iterator (like list slicing but for any iterator)
all_kmers = kmer_generator("ATGCGATCGATCGATCGATCG", 4)
first_five = list(islice(all_kmers, 5))
print(f"First 5 of 4-mers: {first_five}")

# Skip first 3, take next 4
all_kmers = kmer_generator("ATGCGATCGATCGATCGATCG", 4)
middle = list(islice(all_kmers, 3, 7))
print(f"4-mers 3 through 6: {middle}")

In [ ]:
# takewhile -- take elements while condition is true
# dropwhile -- skip elements while condition is true, then yield the rest

quality_scores = [38, 40, 39, 37, 35, 20, 15, 10, 5, 3, 2]

# Take high-quality bases from the start
high_qual = list(takewhile(lambda q: q >= 30, quality_scores))
print(f"High-quality prefix: {high_qual}")

# Drop the high-quality prefix, get the rest
low_qual_tail = list(dropwhile(lambda q: q >= 30, quality_scores))
print(f"After quality drops: {low_qual_tail}")

### 9.7 `zip_longest` -- Zip with Fill Value

In [ ]:
from itertools import zip_longest

# Align sequences of different lengths
seq1 = "ATGCGATCGA"
seq2 = "ATGCGA"

print("Alignment with gap fill:")
print(f"  Seq1: {seq1}")
print(f"  Seq2: {seq2}")
print("        ", end="")
for n1, n2 in zip_longest(seq1, seq2, fillvalue='-'):
    print('|' if n1 == n2 else ' ', end='')
print()

---
## 10. Practical Example: Sliding Window GC Content

In [ ]:
def sliding_window_gc(sequence, window_size=10):
    """Generator for sliding window GC content."""
    for i in range(len(sequence) - window_size + 1):
        window = sequence[i:i + window_size]
        gc = sum(1 for nuc in window if nuc in 'GC') / window_size
        yield (i, gc)


dna = "AAATTTGCGCGCGCGCAAATTTAAAGCGCGCGCATATATGCGC"
print(f"Sequence: {dna}")
print(f"{'Pos':>4s}  {'GC%':>5s}  Plot")
print("-" * 40)
for pos, gc in sliding_window_gc(dna, 10):
    bar = '#' * int(gc * 20)
    print(f"{pos:4d}  {gc:5.1%}  {bar}")

---
## 11. Practical Example: Finding ORFs with Generators

In [ ]:
def find_orfs(sequence, min_length=30):
    """Generator that yields ORFs (start-to-stop) in all 3 forward reading frames."""
    stop_codons = {'TAA', 'TAG', 'TGA'}
    
    for frame in range(3):
        i = frame
        while i <= len(sequence) - 3:
            if sequence[i:i+3] == 'ATG':
                # Found a start codon -- scan for stop
                for j in range(i + 3, len(sequence) - 2, 3):
                    codon = sequence[j:j+3]
                    if codon in stop_codons:
                        orf = sequence[i:j+3]
                        if len(orf) >= min_length:
                            yield {
                                'frame': frame + 1,
                                'start': i,
                                'end': j + 3,
                                'length': len(orf),
                                'sequence': orf
                            }
                        i = j + 3  # move past the stop codon
                        break
                else:
                    i += 3
                    continue
                continue
            i += 3


test_dna = "ATGAAAGCCTTTGGGATCGATCGATCGATCGTAAATGCCCCCCGGGATCGATCGATCGTGA"
print(f"Sequence ({len(test_dna)} bp): {test_dna}\n")

for orf in find_orfs(test_dna, min_length=9):
    protein = ''.join(translate_generator(orf['sequence']))
    print(f"Frame +{orf['frame']}: pos {orf['start']}-{orf['end']}, "
          f"{orf['length']} bp -> {protein}")

---
## 12. Practical Example: Pairwise Sequence Comparison

In [ ]:
def pairwise_identity(sequences):
    """Generator yielding pairwise identity for all sequence pairs."""
    for (id1, s1), (id2, s2) in combinations(sequences.items(), 2):
        min_len = min(len(s1), len(s2))
        matches = sum(1 for i in range(min_len) if s1[i] == s2[i])
        identity = matches / min_len if min_len > 0 else 0.0
        yield (id1, id2, identity)


seqs = {
    'Human':      'MVLSPADKTNVKAAWGKVGA',
    'Gorilla':    'MVLSPADKTNVKAAWGKVGA',
    'Mouse':      'MVLSGEDKSNIKAAWGKIGGHAGA',
    'Zebrafish':  'MSLSDKDKAAVKGLWAKISP'
}

print("Hemoglobin alpha-chain pairwise identity:")
for id1, id2, ident in pairwise_identity(seqs):
    print(f"  {id1:12s} vs {id2:12s}: {ident:.1%}")

---
## Exercises

### Exercise 1: Sliding Window Generator

Write a generator `sliding_window(sequence, window_size, step=1)` that yields substrings of `window_size` from a sequence, advancing by `step` bases each time.

Test it on `"ATGCGATCGATCGATCGATCG"` with `window_size=5` and `step=3`.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
def sliding_window(sequence, window_size, step=1):
    """Yield sliding windows of a given size and step."""
    for i in range(0, len(sequence) - window_size + 1, step):
        yield sequence[i:i + window_size]


dna = "ATGCGATCGATCGATCGATCG"
print(f"Windows (size=5, step=3) from {dna}:")
for i, window in enumerate(sliding_window(dna, 5, 3)):
    print(f"  Window {i}: {window}")

### Exercise 2: FASTA Record Streamer

Write a generator `fasta_records(text)` that takes a multi-line FASTA string (not a file) and yields `(gene_id, description, sequence)` tuples.

Test with:
```
>sp|P04637|P53_HUMAN Cellular tumor antigen p53
MEEPQSDPSVEPPLSQETFS
DLWKLLPENNVLSPLPSQAM
>sp|P38398|BRCA1_HUMAN Breast cancer type 1
MDLSALRVEEVQNVINAMQK
ILECPICLELIKEPVSTKCDH
```

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
def fasta_records(text):
    """Parse a FASTA-format string and yield (gene_id, description, sequence) tuples."""
    header = None
    seq_parts = []
    
    for line in text.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if header is not None:
                parts = header.split(None, 1)
                gene_id = parts[0]
                desc = parts[1] if len(parts) > 1 else ''
                yield (gene_id, desc, ''.join(seq_parts))
            header = line[1:]
            seq_parts = []
        else:
            seq_parts.append(line)
    
    if header is not None:
        parts = header.split(None, 1)
        gene_id = parts[0]
        desc = parts[1] if len(parts) > 1 else ''
        yield (gene_id, desc, ''.join(seq_parts))


fasta_text = """>sp|P04637|P53_HUMAN Cellular tumor antigen p53
MEEPQSDPSVEPPLSQETFS
DLWKLLPENNVLSPLPSQAM
>sp|P38398|BRCA1_HUMAN Breast cancer type 1
MDLSALRVEEVQNVINAMQK
ILECPICLELIKEPVSTKCDH
"""

for gene_id, desc, seq in fasta_records(fasta_text):
    print(f"ID:   {gene_id}")
    print(f"Desc: {desc}")
    print(f"Seq:  {seq[:30]}... ({len(seq)} aa)\n")

### Exercise 3: Random DNA Sequence Generator

Write a generator `random_dna(length, gc_content=0.5)` that yields one random nucleotide at a time, producing a DNA sequence of the specified length with approximately the given GC content.

Then use it to generate a 1000-bp sequence with 60% GC content and verify the actual GC content.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
import random


def random_dna(length, gc_fraction=0.5):
    """Yield random nucleotides with specified GC content."""
    gc_half = gc_fraction / 2
    at_half = (1 - gc_fraction) / 2
    nucleotides = ['A', 'T', 'G', 'C']
    weights = [at_half, at_half, gc_half, gc_half]
    
    for _ in range(length):
        yield random.choices(nucleotides, weights=weights)[0]


random.seed(42)
seq = ''.join(random_dna(1000, gc_fraction=0.6))
actual_gc = sum(1 for n in seq if n in 'GC') / len(seq)

print(f"Generated sequence (first 80 bp): {seq[:80]}")
print(f"Length: {len(seq)}")
print(f"Target GC: 60.0%")
print(f"Actual GC: {actual_gc:.1%}")

### Exercise 4: K-mer Frequency Counter Using Generators

Use the `kmer_generator` and `collections.Counter` to find the 10 most common 4-mers in the following sequence. Do this without building a list of all k-mers -- pass the generator directly to `Counter`.

```python
dna = "ATGCGATCGATCGATCGATCGATCGAATTCGGATCCAAGCTTATGCGATCGATCGATCGATCG"
```

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
from collections import Counter

dna = "ATGCGATCGATCGATCGATCGATCGAATTCGGATCCAAGCTTATGCGATCGATCGATCGATCG"

# Pass generator directly to Counter -- no intermediate list
kmer_counts = Counter(kmer_generator(dna, 4))

print(f"Sequence length: {len(dna)} bp")
print(f"Unique 4-mers: {len(kmer_counts)}")
print(f"\nTop 10 most common 4-mers:")
for kmer, count in kmer_counts.most_common(10):
    print(f"  {kmer}: {count}")

### Exercise 5: Codon Usage Table with itertools

Given the `CODON_TABLE` dictionary, use `itertools.groupby` (after sorting) to produce a codon usage summary grouped by amino acid. For each amino acid, show how many codons encode it.

Hint: First create a sorted list of `(amino_acid, codon)` pairs, then use `groupby`.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
from itertools import groupby

# Create sorted (aa, codon) pairs
aa_codon_pairs = sorted(
    ((aa, codon) for codon, aa in CODON_TABLE.items()),
    key=lambda x: x[0]
)

print(f"{'AA':>4s}  {'#Codons':>7s}  Codons")
print("-" * 45)
for aa, group in groupby(aa_codon_pairs, key=lambda x: x[0]):
    codons = [codon for _, codon in group]
    label = 'Stop' if aa == '*' else aa
    print(f"{label:>4s}  {len(codons):>7d}  {', '.join(sorted(codons))}")

### Exercise 6: Restriction Site Fragment Generator

Write a generator `digest(sequence, site)` that simulates a restriction enzyme digestion. It should yield the fragments produced by cutting the sequence at every occurrence of the restriction `site`.

Test with EcoRI (GAATTC) on: `"ATGCGAATTCGATCGATCGAATTCGGATCGATCG"`

Expected fragments: `"ATGCG"`, `"AATTCGATCGATCG"`, `"AATTCGGATCGATCG"`

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
def digest(sequence, site):
    """Simulate restriction enzyme digestion. Cuts before the site.
    
    Yields DNA fragments after cutting at every occurrence of the site.
    """
    start = 0
    pos = sequence.find(site, start)
    
    while pos != -1:
        yield sequence[start:pos]
        start = pos
        pos = sequence.find(site, start + 1)
    
    yield sequence[start:]


dna = "ATGCGAATTCGATCGATCGAATTCGGATCGATCG"
print(f"Sequence: {dna}")
print(f"Enzyme: EcoRI (GAATTC)\n")

print("Fragments:")
for i, fragment in enumerate(digest(dna, 'GAATTC'), 1):
    print(f"  Fragment {i}: {fragment} ({len(fragment)} bp)")

### Exercise 7: Interleave Paired-End Reads

In paired-end sequencing, reads come in two files (R1 and R2). Write a generator `interleave(reads_r1, reads_r2)` that yields records alternating from R1 and R2.

Use it to interleave two lists of read IDs:
```python
r1 = ['read1/1', 'read2/1', 'read3/1']
r2 = ['read1/2', 'read2/2', 'read3/2']
```
Expected: `read1/1, read1/2, read2/1, read2/2, read3/1, read3/2`

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
def interleave(reads_r1, reads_r2):
    """Interleave paired-end reads: R1_1, R2_1, R1_2, R2_2, ..."""
    for r1, r2 in zip(reads_r1, reads_r2):
        yield r1
        yield r2


r1 = ['read1/1', 'read2/1', 'read3/1']
r2 = ['read1/2', 'read2/2', 'read3/2']

print("Interleaved reads:")
for read in interleave(r1, r2):
    print(f"  {read}")

---
## Summary

### Key Concepts

| Concept | Description | When to Use |
|---|---|---|
| **Iterator Protocol** | `__iter__()` + `__next__()` | Custom iteration logic |
| **Generators** (`yield`) | Functions that pause and resume | Large files, streaming data |
| **Generator Expressions** | `(expr for x in iterable)` | One-pass aggregations (sum, count) |
| **itertools** | `chain`, `product`, `combinations`, `groupby`, etc. | Combinatorics, data pipelines |

### Bioinformatics Applications
- **Streaming file readers**: FASTA/FASTQ parsing without loading entire files
- **K-mer generation**: All possible or observed k-mers from sequences
- **Six-frame translation**: Lazy codon iteration over genomes
- **Quality filtering pipelines**: Chain generators for read processing
- **Combinatorial analysis**: All codons, primer pairs, pairwise comparisons

In [ ]:
# Cleanup
import os
for f in ['sample.fastq', 'sample.fasta']:
    if os.path.exists(f):
        os.remove(f)
print("Cleanup complete.")

---

[← Previous: Module 10: Comprehensions](../../Tier_1_Python_for_Bioinformatics/10_Comprehensions/01_comprehensions.ipynb) | [Next: Regular Expressions for Bioinformatics →](../../Tier_1_Python_for_Bioinformatics/12_Regular_Expressions/01_regular_expressions.ipynb)